# Charlotteburg Reservoir contributing watershed and USGS stations

USGS outlet **01382380**. Uses the same NLDI unsimplified upstream-basin method, UTM 18N metric operations, and 2 km outside buffer as Modules 6 and 7. This is not a new DEM delineation. Reservoir transfers are excluded. Published reference drainage area: 56.2 mi². Review the reported difference; do not force a match.

Run from a clone (or clone this repository in Colab first) with `pip install -r requirements.txt`. The executed cells below validate and display the saved snapshot without repeating service queries.

In [ ]:
from pathlib import Path
import sys, json, subprocess
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'scripts/newark_analysis.py').is_file()), None)
if ROOT is None:
    ROOT = (Path('/content') if Path('/content').exists() else Path.cwd()) / 'newark-water-supply'
    if not (ROOT/'scripts/newark_analysis.py').is_file():
        subprocess.run(['git', 'clone', 'https://github.com/JohannVera/newark-water-supply.git', str(ROOT)], check=True)
import pandas as pd, geopandas as gpd
sys.path.insert(0, str(ROOT/'scripts'))
import newark_analysis as analysis
KEY = 'charlotteburg'
FOLDER = ROOT/'data'/KEY


In [ ]:
# Reuse cached basin and recompute its geometry and area checks.
key,basin,validation=analysis.basin(KEY)
print(json.dumps(validation,indent=2))
assert basin.is_valid.all()

In [ ]:
stations=pd.read_csv(FOLDER/'station_analysis.csv',dtype={'site_no':str}).fillna('')
display(stations[['site_no','station_nm','location_type','variable_labels','operational_flag']])
print(stations.location_type.value_counts().to_string())

## Refresh all five reservoirs and consolidate
Uncomment the next cell to query current USGS metadata and seven days of instantaneous values, update the per-watershed station tables and regenerate the consolidated map. This preserves the existing basin files. The pipeline records exact request URLs, UTC times, and raw responses. Re-run the finishing/export steps only when changing the delivered packaging. Observation spacing and latest observation age do not verify transmission frequency or publication latency.

In [ ]:
# analysis.run()

In [ ]:
from IPython.display import display, HTML
import html
map_path = ROOT/'data/consolidated/newark_system_map.html'
map_html = map_path.read_text(encoding='utf-8')
# Also support older local clones with the blocked default tiles.
map_html = map_html.replace('https://tile.openstreetmap.org/{z}/{x}/{y}.png',
                            'https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}.png')
map_html = map_html.replace('</head>', '<!-- Basemap: CARTO; map data: OpenStreetMap contributors --></head>')
display(HTML('<iframe title="Newark reservoir system map" style="width:100%;height:650px;border:0" srcdoc="' +
             html.escape(map_html, quote=True) + '"></iframe>'))
